# Education Project: Modeling Relationships Between ACT Scores and Socio-Economic Factors
### Brisa Halviatti for Data 5100 Assignment 2


**Objective:** The purpose of the assignment is to perform work using the data science methodology to answer the question of whether school performance is predicted by socioeconomic factors. We will perform and compare various linear regression models to determine whether 'less is more': do the additional predictors from additional datasets add significant value to our model or is a simple reduced model the best fit?  
 
**Key Techniques:** Regression Analysis (see [Data_Prep_Education_Project](https://github.com/brisamh/education/blob/main/code/Data_Prep_Education_Project.ipynb) for data definitions and preparation steps)

## Load and Preview Datasource

In [ ]:
# Import pandas, numpy, and matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# data visualization
import seaborn as sns
# set the plotting style
sns.set_style("whitegrid")
import plotly.offline as po
import plotly.graph_objs as pg
po.init_notebook_mode(connected=True)

# Model preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Modeling
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Model metrics and analysis
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.anova import anova_lm
from patsy import dmatrices

Import Prepped Dataset

In [ ]:
df_final = pd.read_csv(
    'https://raw.githubusercontent.com/brisamh/education/refs/heads/main/data/clean_education_project_data.csv')

In [ ]:
df_final.info()

While Zip Code appears to have become an integer, all other data types appear valid

In [ ]:
df_final['zipcode'] = df_final['zipcode'].astype(str)

In [ ]:
pd.set_option('display.max_columns', None)
df_final.head()

## Summary Statistics

In [ ]:
df_final['state'].nunique()

In [ ]:
print(f'There are {len(df_final)} schools in our sample from {df_final['state'].nunique()} states')

The aveage ACT Score is about 20, with a standard deviation of 2.5

In [ ]:
df_final['average_act'].describe()

In [ ]:
layout = dict(
    geo={"scope": "usa"},
    coloraxis_colorbar=dict(title="Number of Schools")
)

data = dict(
    type="choropleth",
    locations=df_final["state"].value_counts().index,
    locationmode="USA-states",
    z=df_final["state"].value_counts().values,
    coloraxis="coloraxis"
)

x = pg.Figure(data=[data], layout=layout)
po.iplot(x)

In [ ]:
state_counts = df_final['state'].value_counts().rename_axis('state').reset_index(name='count')
state_counts['percent'] = (state_counts['count'] / state_counts['count'].sum() * 100).round(2)
state_counts

Texas schools make up almost 13% of the sample, Washington is the only state representing the West Coast, and Wyoming is the only state representing the Midwest. We know that this sample is not going to be the most representative of the entire United States, but can offer some relationships between the relationships in our data

In [ ]:
df_final.groupby('testing')['state'].unique()

Half of the states do not provide or mandate any standardized testing-- it is manadatory in four and free testing is provided in 6.

In [ ]:
df_final['testing'].value_counts(normalize=True).round(3)

Texas, Ohio, and Illinois the states with the most schools represented, and they are testing 'Optional'. This is important to note since we can assume that schools with mandatory testing will have lower reported ACT scores, simply because non-college bound students will also be taking these tests. 

In [ ]:
df_final.groupby('testing')['average_act'].mean()

## Begin Analysis

We will visualize the full relationships present in our data using a pair plot. We can omit some of the extremely sparse or irrelevant data.

In [ ]:
sns.pairplot(
    df_final.drop(columns=['any_institutions', 'any_ref_or_arr'])
)
plt.show()

While not easy to see, there is indication of multicollinearity between some of our predictors: Student Count and Teacher Count have nearly a 1:1 relationship, which makes sense. Referrals and Arrests also have a strong relationship, which again makes complete sense. Teacher Count and area Median Income also appear to have some signal to a relationship. 

Zoom in on the variables for Average ACT, in two blocks:

In [ ]:
fig = sns.pairplot(
    df_final,
    y_vars=['average_act'],
    x_vars=['rate_unemployment',
 'percent_college',
 'percent_married',
 'median_income',
 'average_act',
 'percent_lunch'],  
    kind='reg',
    plot_kws={
        "line_kws": {'color':'blue'},
        "scatter_kws": {"alpha":0.5, "color":"k", "s":7},
    }
)


In [ ]:
fig = sns.pairplot(
    df_final,
    y_vars=['average_act'],
    x_vars=['referrals_or_arrests', 'institution_count','student_cnt','teacher_cnt','student_teacher_ratio'],  
    kind='reg',
    plot_kws={
        "line_kws": {'color':'blue'},
        "scatter_kws": {"alpha":0.5, "color":"k", "s":7},
    }
)

In [ ]:
predictor_variables = ['rate_unemployment', 'percent_college', 'percent_lunch', 'state', 'charter'
                       ,'magnet_sch','shared_time_sch','student_cnt','teacher_cnt','student_teacher_ratio'
                       ,'any_institutions','institution_count','testing','area_type','area_size','any_ref_or_arr','referrals_or_arrests']


numerical_predictors = df_final[predictor_variables].select_dtypes(include='number').columns.to_list()

corr_matrix = df_final[numerical_predictors + ["average_act"]].corr()

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, vmax=1, vmin=-1, square=True, annot=True, cmap="viridis"
)

plt.tick_params(labelsize=12)

plt.show()

#### Try some Categorical Analysis

In [ ]:
fig = sns.pairplot(
    data=df_final,
    vars=numerical_predictors + ['average_act'],
    hue='testing',
    kind="reg",
    plot_kws={"scatter_kws": {"alpha": 0.5, "color": "k", "s": 7},
    },
)
plt.show()

Some indication that scores are distributed differently based on whether tests are mandated or not

In [ ]:
fig = sns.pairplot(
    data=df_final,
    vars=numerical_predictors + ['average_act'],
    hue='any_ref_or_arr',
    kind="reg",
    plot_kws={"scatter_kws": {"alpha": 0.5, "color": "k", "s": 7},
    },
)
plt.show()

Little indication that arrests and referrals influence much, probably due to so few

In [ ]:
fig = sns.pairplot(
    data=df_final,
    vars=numerical_predictors + ['average_act'],
    hue='magnet_sch',
    kind="reg",
    plot_kws={"scatter_kws": {"alpha": 0.5, "color": "k", "s": 7},
    },
)
plt.show()

Again, probably too few "Magnet Schools" to make a significant difference, the distribution of ACT Scores is just slightly shifted but not by much

## Begin Multivariate Regression Model

We are going to start with a full model, then reduce the model until we have struck a balance of explanatory power without over fitting

In [ ]:
multivariate_model = smf.ols(
    formula='average_act ~ rate_unemployment + percent_college + percent_lunch + student_cnt + teacher_cnt + student_teacher_ratio + institution_count + referrals_or_arrests + C(charter) + C(magnet_sch) + C(shared_time_sch) + C(testing) +  C(area_type) + C(area_size) + any_institutions + any_ref_or_arr',
    data=df_final).fit()


In [ ]:
print(multivariate_model.summary())

In [ ]:
y, X_design = dmatrices('average_act ~ rate_unemployment + percent_college + percent_lunch + student_cnt + teacher_cnt + student_teacher_ratio + institution_count + referrals_or_arrests + C(charter) + C(magnet_sch) + C(shared_time_sch) + C(testing) +  C(area_type) + C(area_size) + any_institutions + any_ref_or_arr',
                        data=df_final,
                        return_type='dataframe'
                        )

In [ ]:
X_design.head()

In [ ]:
X = X_design.loc[:, multivariate_model.pvalues<0.05]

print(X.columns)

In [ ]:
multivariate_model_significant = sm.OLS(y, X).fit()

In [ ]:
print(multivariate_model_significant.summary())

To reduce the model further, I will drop Teacher counts since this is half of the student:teacher ratio predictor and the coefficients for them are marginal at best. While statistically significant, I will also drop referrals_or_arrests since the coefficient is tiny at -0.0015, and is correlated to the T/F flag for 'any_ref_or_arr'

In [ ]:
X = X.drop(
    columns=['referrals_or_arrests', 'teacher_cnt']
)

In [ ]:
multivariate_model_significant = sm.OLS(y, X).fit()

In [ ]:
print(multivariate_model_significant.summary())

As one final pass at reducing the model, I will eliminate student_teacher_ratio since the coefficient is again very small, 'and C(area_size)[T.Fringe]' which is on the cusp of no longer being statistically significant

In [ ]:
X = X.drop(
    columns=['student_teacher_ratio','C(area_size)[T.Fringe]'])

multivariate_model_significant = sm.OLS(y, X).fit()

In [ ]:
print(multivariate_model_significant.summary())

In [ ]:
y_hat = multivariate_model_significant.predict()

plt.figure(figsize=(5,5))

plt.plot(y_hat, multivariate_model_significant.resid, 'ko',mec='w')
plt.axhline(0, color='r', linestyle='dashed', lw=2)

plt.xlabel('Predicted Score')
plt.ylabel('Residual Score')

plt.tick_params(labelsize=14)

plt.show()

An ultra reduced model would look like this:

In [ ]:
model_reduced = smf.ols(
    formula='average_act ~ rate_unemployment + percent_college + percent_lunch'
    ,data=df_final).fit()

In [ ]:
print(model_reduced.summary())

In [ ]:
y_hat = model_reduced.predict()

plt.figure(figsize=(5,5))

plt.plot(y_hat, model_reduced.resid, 'ko',mec='w')
plt.axhline(0, color='r', linestyle='dashed', lw=2)

plt.xlabel('Predicted Score')
plt.ylabel('Residual Score')

plt.tick_params(labelsize=14)

plt.show()

In [ ]:
anova_lm(model_reduced,multivariate_model_significant)

We can see here that our residual sum of squares has reduced quite a bit, from 17k to to 15.7k. The P-value for our F Stat is extremely significant, but the F-stat itself is somewhat weak at 10.7. Ultimately, adding 781 degrees of freedom for a F-Stat smaller than 10 tells us the reduced model may still be the better option.